# 第一阶段 步骤10：测试

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第十步。

---

## 核心目标

框架功能基本成型，这一步用 Python 标准库的 **`unittest`** 写测试，确保实现正确、可维护，并为第一阶段收官。

## 10.1 Python 的单元测试（unittest）

用标准库 `unittest`：写一个继承 `unittest.TestCase` 的类，测试方法以 `test` 开头命名。

- 断言用 `self.assertEqual(a, b)` 判断相等；
- 书里用 `python -m unittest steps/step10.py` 运行；notebook 里用 `TextTestRunner` 运行。

In [ ]:
import numpy as np
import unittest

# 承接步骤09：完整框架 + square/exp + 数值微分
def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x

class Variable:
    def __init__(self, data):
        if data is not None and not isinstance(data, np.ndarray):
            raise TypeError(f'{type(data)} 不是支持的 ndarray 类型')
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)
            if x.creator is not None:
                funcs.append(x.creator)

class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(as_array(y))
        output.set_creator(self)
        self.input = input
        self.output = output
        return output

    def forward(self, x):
        raise NotImplementedError()

    def backward(self, gy):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        x = self.input.data
        return 2 * x * gy

class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        return np.exp(x) * gy

def square(x):
    return Square()(x)

def exp(x):
    return Exp()(x)

# 步骤04 的数值微分（用于梯度检验）
def numerical_diff(f, x, eps=1e-4):
    x0 = Variable(x.data - eps)
    x1 = Variable(x.data + eps)
    y0 = f(x0)
    y1 = f(x1)
    return (y1.data - y0.data) / (2 * eps)

## 10.2 square 函数的测试

- **正向**：`square(2.0)` 应为 `4.0`；
- **反向**：`x = 3.0` 处导数 `2x = 6.0`。

预期值都**手动算好**，测试里断言相等。

In [ ]:
# 10.2 正向 + 反向测试
class SquareTest(unittest.TestCase):
    def test_forward(self):
        x = Variable(np.array(2.0))
        y = square(x)
        self.assertEqual(y.data, np.array(4.0))

    def test_backward(self):
        x = Variable(np.array(3.0))
        y = square(x)
        y.backward()
        self.assertEqual(x.grad, np.array(6.0))   # 2x = 6

## 10.3 通过梯度检验（gradient checking）自动测试

手动算预期值只适合简单函数。**梯度检验**把数值微分结果和反向传播结果比较，若相差大说明反向传播实现有 bug，可代替手工推导：

- 用随机输入，反向传播求 `x.grad`；
- 用 `numerical_diff` 求数值梯度；
- `np.allclose(a, b)` 判断两者是否"足够接近"。

In [ ]:
# 10.3 梯度检验
def test_gradient_check(self):
    x = Variable(np.random.rand(1))
    y = square(x)
    y.backward()
    num_grad = numerical_diff(square, x)
    self.assertTrue(np.allclose(x.grad, num_grad))

SquareTest.test_gradient_check = test_gradient_check

# 运行全部测试
result = unittest.TextTestRunner().run(
    unittest.defaultTestLoader.loadTestsFromTestCase(SquareTest)
)

## 这一步的"为什么"

- **测试是框架的"安全网"**：以后改代码、加功能，跑一遍测试就能立刻发现回归；
- **梯度检验是半自动测试**：不用为每个函数手算导数，用数值微分当"标准答案"，可系统性地覆盖更多函数。

---

> 至此，**第一阶段「自动微分」完成**（步骤01～10）：
> `Variable → Function → 计算图 → 反向传播 → 自动化 → 循环优化 → 易用性 → 测试`。
> 下一阶段开始扩展功能（步骤11 起，如支持多输入多输出的函数）。